# Allen Neuropixels Neural Dynamics Exploration
This notebook provides a brief exploration of the Visual Coding Neuropixels dataset, validating our extraction and processing.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import matplotlib.pyplot as plt
from src.utils import load_config
from src.data_access import get_session
from src.preprocessing import extract_units, build_population_activity
from src.features import build_stimulus_signal

In [ ]:
config = load_config('../config/default.yaml')
session = get_session(config['data']['session_id'], '../data/raw')
print("Session loaded.")

In [ ]:
unit_ids = extract_units(session, region_acronym=config['data']['region_acronym'], min_units=10)
print(f"Extracted {len(unit_ids)} units.")

In [ ]:
stim_table = session.stimulus_presentations
t_start = stim_table['start_time'].min()
t_stop = stim_table['stop_time'].max()

time_vector, x_t = build_population_activity(
    session, unit_ids, t_start=t_start, t_stop=t_start+100, 
    bin_size=config['data']['bin_size'], 
    smoothing_sigma=config['data']['smoothing_sigma']
)

_, u_t = build_stimulus_signal(
    session, t_start=t_start, t_stop=t_start+100,
    bin_size=config['data']['bin_size'],
    stimulus_class=config['features']['stimulus_class']
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(time_vector[:1000], x_t[:1000], label='Population Activity x(t)')
plt.fill_between(time_vector[:1000], 0, u_t[:1000]*x_t.max(), color='gray', alpha=0.3, label='Stimulus u(t)')
plt.xlabel('Time (s)')
plt.ylabel('Firing Rate (Hz)')
plt.legend()
plt.show()